# Expansion Queries

### Total ACV Closed - Expansion Opportunities

Identify expansion companies (Closed Won) created at least 30 days after the first Closed Won for a company, where:

- The deal is of type **Procurement**
- It’s **not deleted**
- It’s **not a Renewal**
- It’s **Inside Sales**

**What we want:**
- Individual deal ACVs and Create Dates

---

#### ✅ Raaji’s comment:
- Should dropped opportunities be included?

---


In [1]:
import psycopg2
import pandas as pd

In [3]:
query = """
WITH first_closed_won AS (
  SELECT
    "AccountId",
    "Name",
    MIN("Close_Date__c")::date AS first_close_date
  FROM "Opportunity"
  WHERE "StageName" = 'Closed Won'
    AND "RecordTypeId" = '0121H0000019ox8QAA' -- Procurement
    AND "IsDeleted" = FALSE
    AND "Agent_1_is_Inside_Sales__c" = TRUE
    AND "Renewal__c" = FALSE
  GROUP BY "AccountId", "Name"
),

expansion_opportunities AS (
  SELECT
    o."AccountId",
    o."Name",
    o."CreatedDate"::date AS create_date,
    o."Annual_Contract_Value_ACV__c" AS acv,
    fcw.first_close_date
  FROM "Opportunity" o
  JOIN first_closed_won fcw
    ON o."AccountId" = fcw."AccountId"
  WHERE o."StageName" = 'Closed Won'
    AND o."CreatedDate"::date > fcw.first_close_date + INTERVAL '30 days'
    AND o."RecordTypeId" = '0121H0000019ox8QAA'
    AND o."IsDeleted" = FALSE
    AND o."Agent_1_is_Inside_Sales__c" = TRUE
    AND o."Renewal__c" = FALSE
)

SELECT
  create_date,
  "Name",
  SUM(acv) AS total_acv_closed_expansion
FROM expansion_opportunities
GROUP BY create_date, "Name"
HAVING SUM(acv) > 0
ORDER BY create_date;
"""

try:
    with psycopg2.connect(**conn_params) as conn:
        df = pd.read_sql_query(query, conn)
    print("success")
    display(df)
except Exception as e:
    print("Error:", e)


/var/folders/bv/w7yj3n891xz17bhg11vlgy700000gn/T/ipykernel_42908/3640762990.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


✅ Query executed successfully.


,create_date,Name,total_acv_closed_expansion
0,2019-08-19,405 Main Level Office - Electric (CPT) - 12/20...,99666.70
1,2019-08-19,Blackinton Operating - Electric (National Grid...,9898.06
2,2019-08-19,CA/JSQ 29-33 (West Quad) - Electric (Ameren) -...,8568.01
3,2019-08-19,Campus Acquistions 308 Green(CAMPUS ACQUISITIO...,13183.76
4,2019-08-19,"Carbondale Evolve, LLC - Electric (Ameren) - 7...",6231.55
...,...,...,...
1084,2025-05-19,"Excelsior Medical Corporation - Gas (NJNG, NJ)...",92511.90
1085,2025-05-22,"Medline Industries, Inc - Electric(PGE, CA) - ...",103801.41
1086,2025-06-05,Article Student Living Davis Property Owner LL...,1205.64
1087,2025-06-10,JRC Burton Preservation Limited Dividend Housi...,28912.61


### Total ACV Created (Expansion)

Identify **expansion deals** (Closed Won) created **at least 30 days after** the **first Closed Won** for a company.

#### ✅ Filters:
- Deal type: **Procurement**
- **Not** deleted
- **Not** a renewal
- Must be **Inside Sales**
- Use deal creation date (`CreatedDate`)
- ACV is calculated as: `kWhe__c * 0.0035`

#### 📈 Output:
- Individual deal **ACV created**
- Corresponding **Created Date**
- Grouped and summed by Created Date and Opportunity Name


In [4]:
query = """
WITH first_closed_won AS (
SELECT
   "AccountId",
   MIN("Close_Date__c")::date AS first_close_date,
   "Name"
FROM "Opportunity"
WHERE "StageName" = 'Closed Won'
and "RecordTypeId" = '0121H0000019ox8QAA' -- Procurement
AND "IsDeleted" = False
and "Agent_1_is_Inside_Sales__c"=True
and "Renewal__c" = False
GROUP BY "AccountId", "Name"
),
expansion_opportunities AS (
SELECT
   o."AccountId",
   o."Name",
   o."CreatedDate"::date AS create_date,
   (o."kWhe__c" * 0.0035) AS acv,
   fcw.first_close_date
FROM "Opportunity" o
JOIN first_closed_won fcw
   ON o."AccountId" = fcw."AccountId"
WHERE o."CreatedDate"::date >  fcw.first_close_date + INTERVAL '30 days'
and "RecordTypeId" = '0121H0000019ox8QAA' -- Procurement
AND "IsDeleted" = False
and "Agent_1_is_Inside_Sales__c"=True
and "Renewal__c" = False
)
SELECT
create_date,
"Name",
sum(acv) AS total_acv_created_expansion
FROM expansion_opportunities
group by create_date, "Name"
having sum(acv) is not null --  taking non null values
ORDER BY create_date
"""

try:
    with psycopg2.connect(**conn_params) as conn:
        df = pd.read_sql_query(query, conn)
    print("success")
    display(df)
except Exception as e:
    print("Error:", e)


/var/folders/bv/w7yj3n891xz17bhg11vlgy700000gn/T/ipykernel_42908/9915645.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


success


,create_date,Name,total_acv_created_expansion
0,2019-08-19,405 Main Level Office - Electric (CPT) - 12/20...,67734.6600
1,2019-08-19,Atrium Events LLC - Gas - 06/2019,1439.2000
2,2019-08-19,Biddeford Microlife - Electric 6/2019-,853.7515
3,2019-08-19,Blackinton Operating - Electric (National Grid...,4173.8830
4,2019-08-19,Brass Monkey Inc. - Electric 4/2019-,425.1030
...,...,...,...
2196,2025-06-11,"CANTERBURY STATION, LLC - Electric (PotomacEdi...",2301.2500
2197,2025-06-11,"(CSA) Key Creekside LLC - Electric (Oncor, TX)...",0.0000
2198,2025-06-11,Eva White Redevelopment Limited - Electric(Eve...,5653.4730
2199,2025-06-11,"Washington Rehab Assoc - Electric (Pepco, DC) ...",1165.2900


### Number of Companies That Sent Invoices (Expansion)

Identify **companies** (from all opportunity stages) that created **expansion deals** **at least 30 days** after their **first Closed Won** deal.

#### ✅ Filters:
- Opportunity type: **Procurement**
- **Not deleted**
- **Not a renewal**
- Must be **Inside Sales**
- No restriction on stage (unlike previous “Closed Won” filters)
- Expansion defined as: any deal **Created > 30 days** after first Closed Won

#### 🚫 Note:
- In the future, stages like **‘Waiting on invoices’** and **‘Needs more info’** will be **excluded**

#### 📈 Output:
- Date expansion deal was created
- Company name (Opportunity)
- One row per qualifying expansion opportunity


In [7]:
query = """
WITH first_closed_won AS (
 SELECT
     "AccountId",
     "Company_Name_Text__c",
     MIN("Close_Date__c")::date AS first_close_date
 FROM "Opportunity"
 WHERE  "RecordTypeId" = '0121H0000019ox8QAA' -- Procurement
 AND "IsDeleted" = False
 and "Agent_1_is_Inside_Sales__c"=True
 and "Renewal__c" = False
 GROUP BY "AccountId", "Company_Name_Text__c"
),
expansion_opportunities AS (
 SELECT
     o."AccountId",
     o."Company_Name_Text__c",
     o."CreatedDate"::date AS create_date,
     o."Annual_Contract_Value_ACV__c" AS acv,
     fcw.first_close_date
 FROM "Opportunity" o
 JOIN first_closed_won fcw
     ON o."AccountId" = fcw."AccountId"
 WHERE o."CreatedDate"::date >  fcw.first_close_date + INTERVAL '30 Days'
 and "RecordTypeId" = '0121H0000019ox8QAA' -- Procurement
 AND "IsDeleted" = False
 and "Agent_1_is_Inside_Sales__c"=True
 and "Renewal__c" = False)


SELECT create_date,
      "Company_Name_Text__c"
FROM expansion_opportunities
group by create_date, "Company_Name_Text__c"
ORDER BY create_date desc
"""

try:
    with psycopg2.connect(**conn_params) as conn:
        df = pd.read_sql_query(query, conn)
    print("success")
    display(df)
except Exception as e:
    print("Error:", e)


/var/folders/bv/w7yj3n891xz17bhg11vlgy700000gn/T/ipykernel_42908/966301551.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


success


,create_date,Name
0,2025-06-11,60 Kilmarnock (Boston) Owner LLC - Electric(Ev...
1,2025-06-11,Bedford Gardens Redevelopment - Electric (Ever...
2,2025-06-11,"Bedford Gardens Redevelopment LLC - Gas (CNG, ..."
3,2025-06-11,"CANTERBURY STATION, LLC - Electric (PotomacEdi..."
4,2025-06-11,"(CSA) Key Creekside LLC - Electric (Oncor, TX)..."
...,...,...
2965,2019-08-19,The Residences At The VIC Cond. Ass. - Gas 6/2...
2966,2019-08-19,Veloccis Trinity LP Gas (CO)
2967,2019-08-19,"Village Restuarant, Inc Gas 1/2019"
2968,2019-08-19,We Yogis LLC - Electricity - 07/2019


### Meetings Had (Filtered by Activity Type)

Count of **meetings had** based on specific **HubSpot activity types**, grouped by the **created date** of the activity.

#### ✅ Included Activity Types:
- In Person
- Exploratory
- Results (Expansion)
- Results (Results New)
- Expansion

#### 📊 Output:
- **Date** of the activity
- **Activity type**
- **Number of meetings** on that day and type


In [ ]:
query = """
SELECT
 "createdAt" / 1000 as created_at,
 "activityType",
 COUNT(*) AS activity_count
FROM
 newdb.hubspot.engagements
WHERE
 "activityType" IN (
   'In Person',
   'Exploratory',
   'Results (Expansion)',
   'Results',
   'Expansion'
 )
GROUP BY
 "createdAt",
 "activityType"
ORDER BY
"createdAt" DESC,
 activity_count DESC
"""

try:
    with psycopg2.connect(**conn_params) as conn:
        df = pd.read_sql_query(query, conn)
    print("success")
    display(df)
except Exception as e:
    print("Error:", e)
